In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
data = pd.read_csv('all_tictactoe_games.csv')

In [3]:
y = data['result']
y.value_counts()

result
X win    131184
O win     77904
draw      46080
Name: count, dtype: int64

In [4]:
X = data[['move1', 'move2', 'move3']]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

In [5]:
dummy = DummyClassifier()
print('Dummy accuracy:' , cross_val_score(dummy, X, y, cv=cv).mean())
print('Dummy macro F1:' , cross_val_score(dummy, X, y, cv=cv, scoring='f1_macro').mean())

Dummy accuracy: 0.5141083521444695
Dummy macro F1: 0.22636352341905827


In [15]:
def make_pipeline(columns):
    preprocess = ColumnTransformer([('ohe', OneHotEncoder(handle_unknown='ignore'), columns)])
    return Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=500))])

In [16]:
pipe = make_pipeline(X.columns)

In [17]:
acc = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
f1 = cross_val_score(pipe, X, y, cv=cv, scoring='f1_macro')
print('Accuracy:', acc.mean())
print('Macro F1:', f1.mean())
pipe.fit(X, y)


Accuracy: 0.516647855530474
Macro F1: 0.2854638860010791


,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('ohe', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
all_moves = [f'move{i}' for i in range(1, 10)]
X_all = data[all_moves]
pipe_all = make_pipeline(X_all.columns)
acc_all = cross_val_score(pipe_all, X_all, y, cv=cv, scoring='accuracy')
f1_all = cross_val_score(pipe_all, X_all, y, cv=cv, scoring='f1_macro')
print('All-move accuracy:', acc_all.mean())
print('All-move macro F1:', f1_all.mean())


All-move accuracy: 0.8284424379232506
All-move macro F1: 0.7308859526903295


In [13]:
prep_step = pipe.named_steps['prep']
feature_names = prep_step.get_feature_names_out()
print(feature_names)

['ohe__move1_C' 'ohe__move1_E' 'ohe__move1_N' 'ohe__move1_NE'
 'ohe__move1_NW' 'ohe__move1_S' 'ohe__move1_SE' 'ohe__move1_SW'
 'ohe__move1_W' 'ohe__move2_C' 'ohe__move2_E' 'ohe__move2_N'
 'ohe__move2_NE' 'ohe__move2_NW' 'ohe__move2_S' 'ohe__move2_SE'
 'ohe__move2_SW' 'ohe__move2_W' 'ohe__move3_C' 'ohe__move3_E'
 'ohe__move3_N' 'ohe__move3_NE' 'ohe__move3_NW' 'ohe__move3_S'
 'ohe__move3_SE' 'ohe__move3_SW' 'ohe__move3_W']


In [18]:
classifier = pipe.named_steps['clf']
for class_name, coefficients in zip(classifier.classes_, classifier.coef_):
    strengths = pd.Series(index=feature_names, data=coefficients)
    print(class_name)
    print("Strongest positive weights")
    print(strengths.nlargest(10))
    print("Strongest negative weights")
    print(strengths.nsmallest(10))


O win
Strongest positive weights
ohe__move2_C     0.199515
ohe__move1_W     0.124015
ohe__move1_E     0.124015
ohe__move3_W     0.124015
ohe__move1_S     0.124015
ohe__move3_E     0.124015
ohe__move1_N     0.124015
ohe__move3_N     0.124015
ohe__move3_S     0.124015
ohe__move2_NW    0.068969
dtype: float64
Strongest negative weights
ohe__move3_C    -0.266952
ohe__move1_C    -0.266952
ohe__move2_S    -0.124139
ohe__move2_E    -0.124139
ohe__move2_N    -0.124139
ohe__move2_W    -0.124139
ohe__move1_SW   -0.062568
ohe__move3_NE   -0.062568
ohe__move1_SE   -0.062568
ohe__move1_NW   -0.062568
dtype: float64
X win
Strongest positive weights
ohe__move1_C     0.255858
ohe__move3_C     0.255858
ohe__move2_N     0.154576
ohe__move2_W     0.154576
ohe__move2_S     0.154576
ohe__move2_E     0.154576
ohe__move1_NW    0.069149
ohe__move1_NE    0.069149
ohe__move3_SE    0.069149
ohe__move3_SW    0.069149
dtype: float64
Strongest negative weights
ohe__move2_C    -0.212240
ohe__move1_E    -0.032738
ohe

In [19]:
importance_data = data.sample(n=20000, random_state=0)
importance = permutation_importance(pipe,
                                    importance_data[['move1', 'move2', 'move3']],
                                    importance_data['result'],
                                    scoring='f1_macro',
                                    n_repeats=5,
                                    random_state=0)
move_importance = pd.Series(index=X.columns, data=importance.importances_mean)
print(move_importance.sort_values(ascending=False))


move2    0.024116
move3    0.008623
move1    0.007623
dtype: float64
